# Procedural Memory | Agent Memory System

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import Dict, List, Optional
from dataclasses import dataclass, field

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Procedural Memory for an Operations Agent

@dataclass
class Procedure:
    name: str
    trigger_pattern: str
    steps: List[str]
    success_rate: float = 1.0
    times_used: int = 0

class ProceduralMemory:
    def __init__(self):
        self.procedures: Dict[str, Procedure] = {}

    def store(self, proc: Procedure):
        self.procedures[proc.name] = proc

    def find(self, task_description: str) -> Optional[Procedure]:
        if not self.procedures:
            return None
        # Fast path: exact keyword match (avoids LLM call)
        task_lower = task_description.lower()
        for name, proc in self.procedures.items():
            if any(kw in task_lower for kw in proc.trigger_pattern.lower().split(", ")):
                return proc
        # Slow path: LLM-based fuzzy matching (handles paraphrases)
        proc_list = "\n".join(
            f"- {name}: {p.trigger_pattern}" for name, p in self.procedures.items()
        )
        response = model.invoke(
            f"Which procedure matches this task?\n\nTask: {task_description}\n\n"
            f"Procedures:\n{proc_list}\n\nRespond with ONLY the procedure name, or 'none'."
        )
        return self.procedures.get(response.content.strip().lower())

In [4]:
memory = ProceduralMemory()
memory.store(Procedure(
    name="deploy_service",
    trigger_pattern="Deploying a service or pushing to production",
    steps=["Run tests", "Build Docker image", "Push to registry", "Update K8s manifest",
           "Apply rollout", "Monitor health 5min", "Rollback if unhealthy"],
))
memory.store(Procedure(
    name="debug_api_error",
    trigger_pattern="Debugging API errors, 4xx or 5xx responses",
    steps=["Check application logs", "Identify error and stack trace", "Check recent deployments",
           "Verify DB connectivity", "Check external dependencies", "Reproduce locally", "Fix and add test"],
))

task = "We need to push the new payment feature to production"
proc = memory.find(task)
if proc:
    print(f"Found: {proc.name}\nSteps:")
    for step in proc.steps:
        print(f"  - {step}")

Found: deploy_service
Steps:
  - Run tests
  - Build Docker image
  - Push to registry
  - Update K8s manifest
  - Apply rollout
  - Monitor health 5min
  - Rollback if unhealthy


## Extended: Hierarchical Procedural Memory

In [5]:
# Hierarchical Procedural Memory

@dataclass
class HierProc:
    name: str
    description: str
    level: str  # "strategy", "procedure", "action"
    steps: List[str] = field(default_factory=list)
    children: List[str] = field(default_factory=list)

class HierarchicalProceduralMemory:
    def __init__(self):
        self.procedures: Dict[str, HierProc] = {}

    def add(self, proc: HierProc):
        self.procedures[proc.name] = proc

    def expand(self, name: str, indent: int = 0) -> str:
        proc = self.procedures.get(name)
        if not proc:
            return f"{'  ' * indent}[Unknown: {name}]"
        prefix = "  " * indent
        lines = [f"{prefix}{'#' * (indent + 1)} {proc.name} ({proc.level}): {proc.description}"]
        for step in proc.steps:
            lines.append(f"{prefix}  - {step}")
        for child in proc.children:
            lines.append(self.expand(child, indent + 1))
        return "\n".join(lines)

In [6]:
mem = HierarchicalProceduralMemory()
mem.add(HierProc("incident_response", "Full incident workflow", "strategy",
                  children=["diagnose", "mitigate", "post_mortem"]))
mem.add(HierProc("diagnose", "Identify root cause", "procedure",
                  children=["check_logs", "check_metrics"]))
mem.add(HierProc("mitigate", "Apply fix and restore service", "procedure",
                  steps=["Apply fix", "Verify recovery", "Monitor 15 min"]))
mem.add(HierProc("post_mortem", "Document and learn", "procedure",
                  steps=["Write incident report", "Identify improvements", "Update runbooks"]))
mem.add(HierProc("check_logs", "Review application logs", "action",
                  steps=["Access logging platform", "Filter by timestamp", "Look for stack traces"]))
mem.add(HierProc("check_metrics", "Review system metrics", "action",
                  steps=["Check CPU/memory/disk", "Review latency graphs", "Check error rates"]))

print(mem.expand("incident_response"))

# incident_response (strategy): Full incident workflow
  ## diagnose (procedure): Identify root cause
    ### check_logs (action): Review application logs
      - Access logging platform
      - Filter by timestamp
      - Look for stack traces
    ### check_metrics (action): Review system metrics
      - Check CPU/memory/disk
      - Review latency graphs
      - Check error rates
  ## mitigate (procedure): Apply fix and restore service
    - Apply fix
    - Verify recovery
    - Monitor 15 min
  ## post_mortem (procedure): Document and learn
    - Write incident report
    - Identify improvements
    - Update runbooks
